# Live YOLO segmentation — FLIR camera (OpenCV window)

Run the **Pecan** segmentor on a live camera feed and show masks + class labels in a plain **OpenCV** window.

## Python environment

| Kernel | Python | PySpin (FLIR SDK) |
|---|---|---|
| Main `.venv` | 3.12 | No |
| `Pecan PySpin (Python 3.10)` | 3.10 | Yes |

For the FLIR Spinnaker camera, use kernel **`Pecan PySpin (Python 3.10)`** (run `powershell -File scripts/setup_spinnaker_env.ps1` once if missing).

For a generic USB webcam, the main **3.12** kernel is fine — set `CAMERA_BACKEND = "opencv"`.

**Controls:** press **`q`** or **Esc** in the video window to stop.

Run cells top to bottom. The live-view cell blocks until you quit.

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

from napari_pecan_py.widgets.yolo_seg.model import (
    inference_imgsz,
    resolve_yolo_device,
    to_yolo_predict_source,
)

# --- model ---
WEIGHTS_PATH = Path(
    r"C:\Users\salir\Desktop\pecan_py_napari\yolo_seg_runs\Pecan Segmentor on Meyer DB - [Pecan].pt"
)
CONF = 0.25
DEVICE = "auto"  # auto | cpu | 0 (first CUDA GPU)

# --- camera ---
CAMERA_BACKEND = "spinnaker"  # "spinnaker" | "opencv"
CAMERA_INDEX = 0
SPINNAKER_CAMERA_INDEX = 0
SPINNAKER_ROOT = Path(r"C:\Program Files\Teledyne\Spinnaker")

# --- display / timing ---
WINDOW_NAME = "Pecan YOLO live"
INFER_EVERY_N_FRAMES = 2  # run YOLO every N frames (1 = every frame)

print(f"Python {sys.version.split()[0]}")
print(f"Weights exist: {WEIGHTS_PATH.exists()}")

Python 3.10.18
Weights exist: True


In [2]:
def require_pyspin() -> None:
    """Verify Teledyne Spinnaker PySpin is available in this kernel."""
    wheel_dir = SPINNAKER_ROOT / "PySpin"
    wheels = sorted(wheel_dir.glob("spinnaker_python-*.whl"))
    wheel_tags = [w.name.split("-")[4] for w in wheels if len(w.name.split("-")) > 4]
    py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"

    try:
        import PySpin

        if not hasattr(PySpin, "System"):
            raise ImportError("wrong PySpin package (PyPI pyspin vs Spinnaker SDK)")
        ver = PySpin.System.GetInstance().GetLibraryVersion()
        print(f"PySpin OK — Spinnaker {ver.major}.{ver.minor}.{ver.type}.{ver.build} (Python {py_ver})")
    except ImportError as exc:
        raise RuntimeError(
            f"PySpin not available (Python {py_ver}).\n"
            f"Spinnaker wheels on disk: {wheel_tags or ['none']}\n\n"
            "Switch kernel to 'Pecan PySpin (Python 3.10)' or run:\n"
            "  powershell -File scripts/setup_spinnaker_env.ps1\n\n"
            "Or set CAMERA_BACKEND = 'opencv' for a generic webcam."
        ) from exc


if CAMERA_BACKEND == "spinnaker":
    require_pyspin()
else:
    print("Skipping PySpin check (opencv backend).")

PySpin OK — Spinnaker 4.2.0.88 (Python 3.10)


In [3]:
class LiveCamera:
    """Live camera via Spinnaker (FLIR) or OpenCV DirectShow."""

    def __init__(self, backend: str, *, index: int = 0, spinnaker_index: int = 0):
        self.backend = backend
        self.index = index
        self.spinnaker_index = spinnaker_index
        self._cap = None
        self._system = None
        self._cam_list = None
        self._cam = None
        self._processor = None

    def open(self) -> None:
        if self.backend == "opencv":
            self._cap = cv2.VideoCapture(self.index, cv2.CAP_DSHOW)
            if not self._cap.isOpened():
                raise RuntimeError(f"OpenCV could not open camera index {self.index}")
            return

        if self.backend == "spinnaker":
            import PySpin

            self._system = PySpin.System.GetInstance()
            self._cam_list = self._system.GetCameras()
            n = self._cam_list.GetSize()
            if n == 0:
                self._release_spinnaker()
                raise RuntimeError("No Spinnaker cameras detected.")
            if self.spinnaker_index >= n:
                self._release_spinnaker()
                raise IndexError(f"Spinnaker camera index {self.spinnaker_index} >= {n}")

            self._cam = self._cam_list[self.spinnaker_index]
            self._cam.Init()
            nodemap = self._cam.GetNodeMap()

            stream_nodemap = self._cam.GetTLStreamNodeMap()
            node_mode = PySpin.CEnumerationPtr(stream_nodemap.GetNode("StreamBufferHandlingMode"))
            if PySpin.IsReadable(node_mode) and PySpin.IsWritable(node_mode):
                newest = node_mode.GetEntryByName("NewestOnly")
                if PySpin.IsReadable(newest):
                    node_mode.SetIntValue(newest.GetValue())

            acq_mode = PySpin.CEnumerationPtr(nodemap.GetNode("AcquisitionMode"))
            if PySpin.IsReadable(acq_mode) and PySpin.IsWritable(acq_mode):
                continuous = acq_mode.GetEntryByName("Continuous")
                if PySpin.IsReadable(continuous):
                    acq_mode.SetIntValue(continuous.GetValue())

            self._processor = PySpin.ImageProcessor()
            self._processor.SetColorProcessing(
                PySpin.SPINNAKER_COLOR_PROCESSING_ALGORITHM_HQ_LINEAR
            )
            self._cam.BeginAcquisition()
            return

        raise ValueError(f"Unknown backend: {self.backend}")

    def read_rgb(self) -> np.ndarray | None:
        if self.backend == "opencv":
            ok, frame_bgr = self._cap.read()
            if not ok:
                return None
            return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        if self.backend == "spinnaker":
            import PySpin

            image_result = self._cam.GetNextImage(1000)
            try:
                if image_result.IsIncomplete():
                    return None
                converted = self._processor.Convert(image_result, PySpin.PixelFormat_RGB8)
                return np.asarray(converted.GetNDArray(), dtype=np.uint8).copy()
            finally:
                image_result.Release()

        return None

    def _release_spinnaker(self) -> None:
        if self._cam is not None:
            try:
                self._cam.EndAcquisition()
            except Exception:
                pass
            try:
                self._cam.DeInit()
            except Exception:
                pass
            del self._cam
            self._cam = None
        if self._cam_list is not None:
            self._cam_list.Clear()
            del self._cam_list
            self._cam_list = None
        if self._system is not None:
            self._system.ReleaseInstance()
            del self._system
            self._system = None
        self._processor = None

    def close(self) -> None:
        if self._cap is not None:
            self._cap.release()
            self._cap = None
        if self.backend == "spinnaker":
            self._release_spinnaker()


def list_spinnaker_cameras() -> list[str]:
    """List cameras without Init/ReleaseInstance (safe to call before opening)."""
    try:
        import PySpin
    except ImportError:
        return []

    system = PySpin.System.GetInstance()
    cam_list = system.GetCameras()
    names: list[str] = []
    try:
        for i in range(cam_list.GetSize()):
            cam = cam_list[i]
            try:
                nodemap = cam.GetTLDeviceNodeMap()
                model_node = PySpin.CStringPtr(nodemap.GetNode("DeviceModelName"))
                serial_node = PySpin.CStringPtr(nodemap.GetNode("DeviceSerialNumber"))
                model = model_node.GetValue() if PySpin.IsReadable(model_node) else "?"
                serial = serial_node.GetValue() if PySpin.IsReadable(serial_node) else "?"
                names.append(f"[{i}] {model} (SN {serial})")
            except Exception as exc:
                names.append(f"[{i}] <error: {exc}>")
            finally:
                del cam
    finally:
        cam_list.Clear()
        del cam_list
    return names


def list_opencv_cameras(max_index: int = 4) -> list[str]:
    found: list[str] = []
    for i in range(max_index):
        cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
        if cap.isOpened():
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            found.append(f"[{i}] {w}x{h}")
        cap.release()
    return found


print("Spinnaker cameras:", list_spinnaker_cameras() or ["(none)"])
print("OpenCV cameras:", list_opencv_cameras() or ["(none)"])

Spinnaker cameras: ['[0] Blackfly S BFS-U3-51S5C (SN 21275118)']
OpenCV cameras: ['[0] 640x480']


In [4]:
assert WEIGHTS_PATH.exists(), f"Weights not found: {WEIGHTS_PATH}"

resolved_device = resolve_yolo_device(DEVICE)
model = YOLO(str(WEIGHTS_PATH))
class_names = {int(k): str(v) for k, v in model.names.items()}

print(f"Classes: {class_names}")
print(f"Device: {resolved_device}")

Classes: {0: 'Pecan'}
Device: 0


In [ ]:
def run_live_view() -> None:
    """Blocking OpenCV loop. Press q or Esc in the window to stop."""
    camera = LiveCamera(
        CAMERA_BACKEND,
        index=CAMERA_INDEX,
        spinnaker_index=SPINNAKER_CAMERA_INDEX,
    )
    camera.open()
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    frame_idx = 0
    display_bgr: np.ndarray | None = None
    t0 = time.perf_counter()
    frames = 0

    print(f"Live view started — press q or Esc in '{WINDOW_NAME}' to stop.")

    try:
        while True:
            rgb = camera.read_rgb()
            if rgb is None:
                if cv2.waitKey(1) & 0xFF in (ord("q"), 27):
                    break
                continue

            frame_idx += 1
            frames += 1
            display_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            if frame_idx % INFER_EVERY_N_FRAMES == 0:
                h, w = rgb.shape[:2]
                results = model.predict(
                    to_yolo_predict_source(rgb),
                    imgsz=inference_imgsz(h, w, model),
                    conf=CONF,
                    device=resolved_device,
                    retina_masks=True,
                    verbose=False,
                )
                if results:
                    display_bgr = results[0].plot()

            elapsed = time.perf_counter() - t0
            if elapsed >= 2.0:
                fps = frames / elapsed
                cv2.setWindowTitle(WINDOW_NAME, f"{WINDOW_NAME} — {fps:.1f} fps")
                frames = 0
                t0 = time.perf_counter()

            cv2.imshow(WINDOW_NAME, display_bgr)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                break
    finally:
        camera.close()
        cv2.destroyAllWindows()
        print("Stopped.")


run_live_view()

Live view started — press q or Esc in 'Pecan YOLO live' to stop.
